# Milestone 2 — Conditional Generation & Tuning (Hemanth)

**Data:** the team's curated COCO 2017 subset ([`kudupudinikhil/coco-processed`](https://www.kaggle.com/datasets/kudupudinikhil/coco-processed)) ·
**Generator:** Stable Diffusion v1.5 · **Text encoder:** CLIP ViT-L/14 · **Evaluator:** CLIP ViT-L/14 (text + image towers) ·
**Runtime:** Google Colab, T4 GPU (≈30–40 min end to end)

[Open in Colab](https://colab.research.google.com/github/hemanthsrinivas1/Text_to_Image_Generation/blob/milestone-2-hemanth/src/notebooks/milestone_2_H.ipynb)

| Milestone 2 requirement | Where |
|---|---|
| Integrate the text encoder with the diffusion model | §4 — captions encoded by hand, passed as `prompt_embeds`, checked against `prompt=` |
| Run baseline conditional generation | §6 — PNDM, 50 steps, CFG 7.5 on 10 fixed prompts |
| Tune classifier-free guidance | §7 — CFG 1 → 15, plus `guidance_rescale` at high CFG |
| Tune the sampler / noise schedule | §8–§10 — 7 samplers, step counts, `timestep_spacing`, Karras sigmas, zero-SNR, β-schedule |
| Negative prompts | §11 — quality negative and a style negative |
| Quality vs. speed | §12 — every run on one CLIP-score / seconds-per-image chart |
| Record the first set of generated images | every run saves its PNGs and a grid; §13 compares baseline and best |

**Protocol.** Every configuration generates the *same 10 prompts from the same starting latents*, so the
only thing that changes between runs is the setting under test. Each run is scored with two CLIP metrics:

- **CLIP-T** — cosine similarity between the generated image and its caption (prompt adherence).
- **CLIP-I** — cosine similarity between the generated image and the *real* COCO photo for that caption
  (how close the picture is to what the caption actually described).

The real photos' own CLIP-T is reported as a reference ceiling.

---
## 1. Setup

Runtime → Change runtime type → **T4 GPU**. On Colab the cell below installs the few packages Colab does not
ship with (PyTorch is already there).

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
SMOKE_TEST = os.environ.get("M2_SMOKE_TEST") == "1"   # tiny model + 2 prompts, used to test the code off-GPU

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "diffusers>=0.31", "transformers>=4.46", "accelerate", "kagglehub", "scipy", "ftfy"],
                   check=True)

In [ ]:
import json, time, random, shutil, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import diffusers, transformers
from diffusers import (StableDiffusionPipeline, PNDMScheduler, DDIMScheduler, DDPMScheduler,
                       LMSDiscreteScheduler, EulerDiscreteScheduler, EulerAncestralDiscreteScheduler,
                       DPMSolverMultistepScheduler, UniPCMultistepScheduler)
from transformers import CLIPModel, CLIPProcessor

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()
diffusers.logging.set_verbosity_error()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SD_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

VERSIONS = {
    "python": sys.version.split()[0],
    "platform": f"{platform.system()} {platform.machine()}",
    "torch": torch.__version__,
    "diffusers": diffusers.__version__,
    "transformers": transformers.__version__,
    "device": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
}
for k, v in VERSIONS.items():
    print(f"{k:<13}: {v}")
if DEVICE.type != "cuda" and not SMOKE_TEST:
    print("\nNo GPU found. Switch the runtime to a T4 GPU; on CPU this notebook takes many hours.")

### Configuration

Every knob the notebook turns lives here.

In [ ]:
SEED = 42
N_PROMPTS = 10                       # fixed prompt set, shared by every run
PROMPT_MIN_WORDS, PROMPT_MAX_WORDS = 8, 18
BATCH = 5                            # images per pipeline call (10 with CFG's unconditional half) — fits a T4

SD_REPOS = ["stable-diffusion-v1-5/stable-diffusion-v1-5", "sd-legacy/stable-diffusion-v1-5"]
EVAL_MODEL = "openai/clip-vit-large-patch14"
KAGGLE_DATASET = "kudupudinikhil/coco-processed"

BASELINE = dict(sampler="PNDM", steps=50, cfg=7.5)        # SD 1.5's default pipeline settings
TUNE_SAMPLER, TUNE_STEPS = "DPM++ 2M", 25                  # the fast sampler used for the CFG / negative sweeps
CFG_GRID = [1.0, 3.0, 5.0, 7.5, 10.0, 12.5, 15.0]
RESCALE_CFGS, GUIDANCE_RESCALE = [12.5, 15.0], 0.7
SAMPLER_STEPS = {"PNDM": 50, "DDIM": 50, "LMS": 30, "Euler": 30, "Euler-a": 30, "DPM++ 2M": 20, "UniPC": 20}
STEP_GRID = [10, 15, 20, 30, 50]
STEP_SAMPLERS = ["DPM++ 2M", "DDIM"]
SCHEDULE_SAMPLER, SCHEDULE_STEPS = "Euler", 20
NEGATIVE_PROMPTS = {
    "quality": "blurry, low quality, low resolution, distorted, deformed, disfigured, watermark, text, jpeg artifacts",
    "style":   "photo, photograph, photorealistic, realistic",      # steers *away* from photorealism
}
DDPM_STEPS, DDPM_PROMPTS = 1000, 2

OUT_DIR = Path(os.environ.get("M2_OUT_DIR", "outputs/hemanth_m2"))

if SMOKE_TEST:   # same code path, tiny sizes — for checking the notebook runs, not for results
    N_PROMPTS, BATCH, DDPM_STEPS = 2, 2, 10
    SD_REPOS = ["hf-internal-testing/tiny-stable-diffusion-torch"]
    EVAL_MODEL = "openai/clip-vit-base-patch32"
    BASELINE["steps"] = 4
    TUNE_STEPS, SCHEDULE_STEPS = 4, 4
    SAMPLER_STEPS = {k: 4 for k in SAMPLER_STEPS}
    STEP_GRID = [3, 5]
    CFG_GRID = [1.0, 7.5, 15.0]
    RESCALE_CFGS = [15.0]

GEN_DIR, PLOT_DIR = OUT_DIR / "generations", OUT_DIR / "plots"
for d in (GEN_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("outputs ->", OUT_DIR.resolve())

---
## 2. Dataset — the team's curated COCO subset

The Kaggle dataset holds the Milestone 1 output: `pairs.parquet` (6,000 captions = 3,000 images × 2), the
512×512 image cache and the stored embeddings. It is found in this order:

1. `T2I_PROCESSED_DIR` if set (a local copy, or a folder on Google Drive),
2. an attached Kaggle input (`/kaggle/input/*/processed`) when running on Kaggle,
3. otherwise downloaded with `kagglehub` (~390 MB). It is a public dataset; if `kagglehub` asks you to log in,
   run `import kagglehub; kagglehub.login()` and paste your Kaggle username and API key.

In [ ]:
def find_processed_dir():
    if os.environ.get("T2I_PROCESSED_DIR"):
        return Path(os.environ["T2I_PROCESSED_DIR"])
    hits = sorted(Path("/kaggle/input").glob("*/**/pairs.parquet")) if Path("/kaggle/input").exists() else []
    if hits:
        return hits[0].parent
    import kagglehub
    root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    return next(root.rglob("pairs.parquet")).parent


PROCESSED = find_processed_dir()
pairs = pd.read_parquet(PROCESSED / "pairs.parquet")
# The manifest stores paths from the machine that built it; rebuild them from the file name.
pairs["img_local"] = pairs["file_name"].map(lambda f: PROCESSED / "images_512" / f)
missing = (~pairs["img_local"].map(Path.is_file)).sum()

print(f"dataset   : {PROCESSED}")
print(f"pairs     : {len(pairs):,}  ({pairs['image_id'].nunique():,} images)")
print(f"missing   : {missing} images")
assert missing == 0, "image cache incomplete"
pairs[["image_id", "cap_rank", "caption", "n_words"]].head()

### The fixed prompt set

10 primary captions (`cap_rank == 0`, so one per image) of 8–18 words — long enough to describe a scene rather
than a single object — drawn with the fixed seed. Every run below generates exactly these.

In [ ]:
pool = pairs[(pairs["cap_rank"] == 0) & pairs["n_words"].between(PROMPT_MIN_WORDS, PROMPT_MAX_WORDS)]
prompts_df = pool.sample(N_PROMPTS, random_state=SEED).reset_index(drop=True)
PROMPTS = prompts_df["caption"].tolist()
REAL_IMAGES = [Image.open(p).convert("RGB") for p in prompts_df["img_local"]]

for i, p in enumerate(PROMPTS):
    print(f"[{i}] {p}")

In [ ]:
n_show = min(5, N_PROMPTS)
fig, axes = plt.subplots(1, n_show, figsize=(3.2 * n_show, 3.6))
for ax, img, cap in zip(np.atleast_1d(axes), REAL_IMAGES, PROMPTS):
    ax.imshow(img); ax.axis("off")
    ax.set_title("\n".join([cap[i:i + 34] for i in range(0, len(cap), 34)][:3]), fontsize=8)
fig.suptitle("Real COCO photos for the first prompts", fontsize=11)
plt.tight_layout(); plt.savefig(PLOT_DIR / "real_photos.png", dpi=110); plt.show()

---
## 3. Models

Stable Diffusion v1.5 in fp16 (≈4 GB) and the CLIP ViT-L/14 evaluator (≈1.7 GB). The safety checker is disabled:
it is a separate model that blacks out images it flags, which would corrupt the scores on harmless COCO prompts.

In [ ]:
pipe = None
for repo in SD_REPOS:
    try:
        pipe = StableDiffusionPipeline.from_pretrained(repo, torch_dtype=SD_DTYPE,
                                                       safety_checker=None, requires_safety_checker=False)
        SD_REPO = repo
        break
    except Exception as exc:
        print(f"could not load {repo}: {type(exc).__name__}")
assert pipe is not None, "no Stable Diffusion repo could be loaded"
pipe = pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=True)
BASE_SCHED_CONFIG = dict(pipe.scheduler.config)
LATENT_SHAPE = (pipe.unet.config.in_channels, pipe.unet.config.sample_size, pipe.unet.config.sample_size)

clip_eval = CLIPModel.from_pretrained(EVAL_MODEL).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(EVAL_MODEL)

print("SD repo       :", SD_REPO)
print("default sched :", pipe.scheduler.__class__.__name__,
      {k: BASE_SCHED_CONFIG[k] for k in ("beta_schedule", "timestep_spacing", "num_train_timesteps")})
print("latent shape  :", LATENT_SHAPE)

---
## 4. Integrating the text encoder by hand

Normally the pipeline hides the text encoder: `pipe(prompt="a dog")` tokenizes and runs CLIP internally. Here we
do that half ourselves — tokenize to 77 tokens, run SD 1.5's CLIP ViT-L/14 text tower, take the per-token hidden
states (`77 × 768`) — and hand the result to the UNet as `prompt_embeds`. No `prompt=` is used anywhere after
this section. The unconditional half of classifier-free guidance is the embedding of the empty string (or of the
negative prompt in §11).

In [ ]:
@torch.no_grad()
def encode_text(texts):
    tok = pipe.tokenizer(texts, padding="max_length", max_length=pipe.tokenizer.model_max_length,
                         truncation=True, return_tensors="pt")
    return pipe.text_encoder(tok.input_ids.to(DEVICE))[0]          # (n, 77, 768) per-token hidden states


PROMPT_EMBEDS = encode_text(PROMPTS)
EMPTY_EMBED = encode_text([""])
print("prompt_embeds :", tuple(PROMPT_EMBEDS.shape), PROMPT_EMBEDS.dtype)

# Check 1: the same as what the pipeline computes internally from `prompt=`. Both sides encode the same 2
# prompts in one batch, and the difference is measured relative to the embeddings' scale: CLIP hidden states reach
# about ±30, where fp16 can only resolve steps of ~0.016, so an absolute tolerance would fail on GPU for rounding alone.
ref_pos, ref_neg = pipe.encode_prompt(PROMPTS[:2], DEVICE, 1, True, None)
ours_pos, ours_neg = encode_text(PROMPTS[:2]), encode_text(["", ""])
abs_diff = max((ref_pos - ours_pos).abs().max().item(), (ref_neg - ours_neg).abs().max().item())
rel_diff = abs_diff / ref_pos.abs().max().item()
print(f"ours vs pipeline: max abs diff {abs_diff:.2e}  ·  relative to embedding scale {rel_diff:.2e}")
assert rel_diff < 1e-2, "hand-built prompt_embeds do not match the pipeline's own encoding"

### Fixed starting latents

Fixing the seed is not enough to make a sweep fair: samplers consume the random-number stream differently, and
ancestral samplers draw fresh noise every step. So the starting noise is generated **once**, on a **CPU**
generator (CUDA and CPU generators give different streams, so a CPU one reproduces on any machine), and passed to
every run. Ancestral samplers also get a freshly seeded CPU generator per call so their per-step noise repeats too.

In [ ]:
LATENTS = torch.randn((N_PROMPTS, *LATENT_SHAPE), generator=torch.Generator("cpu").manual_seed(SEED))
print("latents:", tuple(LATENTS.shape), f"mean {LATENTS.mean():+.3f}  std {LATENTS.std():.3f}")

### Generation, scoring and recording

`run()` is the single entry point every experiment uses: it builds the scheduler from SD 1.5's own config (so a
sampler swap keeps the trained β-schedule unless we override it on purpose), generates the 10 prompts in batches,
times only the pipeline calls, scores the images, saves the PNGs and a grid, and appends a row to `RESULTS`.

In [ ]:
SCHEDULERS = {
    "PNDM": (PNDMScheduler, {}),
    "DDIM": (DDIMScheduler, {}),
    "DDPM": (DDPMScheduler, {}),
    "LMS": (LMSDiscreteScheduler, {}),
    "Euler": (EulerDiscreteScheduler, {}),
    "Euler-a": (EulerAncestralDiscreteScheduler, {}),
    "DPM++ 2M": (DPMSolverMultistepScheduler, {"algorithm_type": "dpmsolver++", "solver_order": 2}),
    "UniPC": (UniPCMultistepScheduler, {}),
}


def make_scheduler(name, **overrides):
    cls, extra = SCHEDULERS[name]
    return cls.from_config(BASE_SCHED_CONFIG, **{**extra, **overrides})


@torch.no_grad()
def clip_image_embeds(images):
    out = []
    for i in range(0, len(images), 16):
        px = clip_proc(images=images[i:i + 16], return_tensors="pt")["pixel_values"].to(DEVICE)
        feats = clip_eval.visual_projection(clip_eval.vision_model(pixel_values=px).pooler_output)
        out.append(torch.nn.functional.normalize(feats.float(), dim=-1))
    return torch.cat(out)


@torch.no_grad()
def clip_text_embeds(texts):
    tok = clip_proc(text=texts, padding="max_length", max_length=77, truncation=True, return_tensors="pt")
    feats = clip_eval.text_projection(clip_eval.text_model(input_ids=tok["input_ids"].to(DEVICE),
                                                           attention_mask=tok["attention_mask"].to(DEVICE)).pooler_output)
    return torch.nn.functional.normalize(feats.float(), dim=-1)


EVAL_TEXT = clip_text_embeds(PROMPTS)
EVAL_REAL = clip_image_embeds(REAL_IMAGES)
REAL_CLIP_T = (EVAL_TEXT * EVAL_REAL).sum(-1).cpu().numpy()
print(f"real photos: CLIP-T {REAL_CLIP_T.mean():.4f} ± {REAL_CLIP_T.std(ddof=1) / np.sqrt(N_PROMPTS):.4f} (s.e.)")


def score(images, idx):
    emb = clip_image_embeds(images)
    clip_t = (emb * EVAL_TEXT[idx]).sum(-1).cpu().numpy()
    clip_i = (emb * EVAL_REAL[idx]).sum(-1).cpu().numpy()
    return clip_t, clip_i


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


RESULTS, IMAGES = [], {}


def run(tag, experiment, sampler, steps, cfg, negative=None, rescale=0.0, sched_kw=None, idx=None):
    """Generate the fixed prompts under one configuration, score, save and record it."""
    idx = list(range(N_PROMPTS)) if idx is None else list(idx)
    pipe.scheduler = make_scheduler(sampler, **(sched_kw or {}))
    neg_embed = EMPTY_EMBED if negative is None else encode_text([negative])

    images, seconds = [], 0.0
    for b in range(0, len(idx), BATCH):
        bi = idx[b:b + BATCH]
        sync(); t0 = time.perf_counter()
        out = pipe(prompt_embeds=PROMPT_EMBEDS[bi],
                   negative_prompt_embeds=neg_embed.expand(len(bi), -1, -1),
                   latents=LATENTS[bi].to(DEVICE, SD_DTYPE),
                   num_inference_steps=steps, guidance_scale=cfg, guidance_rescale=rescale,
                   generator=torch.Generator("cpu").manual_seed(SEED))
        sync(); seconds += time.perf_counter() - t0
        images += out.images

    clip_t, clip_i = score(images, idx)
    run_dir = GEN_DIR / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    for i, img in zip(idx, images):
        img.save(run_dir / f"prompt_{i:02d}.png")

    row = dict(tag=tag, experiment=experiment, sampler=sampler, steps=steps, cfg=cfg,
               guidance_rescale=rescale, negative=negative or "", schedule=json.dumps(sched_kw or {}),
               n=len(idx), sec_per_image=seconds / len(idx),
               clip_t=clip_t.mean(), clip_t_se=clip_t.std(ddof=1) / np.sqrt(len(idx)) if len(idx) > 1 else np.nan,
               clip_i=clip_i.mean(), clip_i_se=clip_i.std(ddof=1) / np.sqrt(len(idx)) if len(idx) > 1 else np.nan)
    RESULTS.append(row)
    IMAGES[tag] = dict(zip(idx, images))
    print(f"{tag:<28} CLIP-T {row['clip_t']:.4f}  CLIP-I {row['clip_i']:.4f}  {row['sec_per_image']:.2f} s/img")
    return row


def show_grid(tags, title, fname, labels=None, n_rows=4, idx=None):
    """Rows = prompts, columns = configurations."""
    idx = list(range(min(n_rows, N_PROMPTS))) if idx is None else list(idx)
    labels = labels or tags
    fig, axes = plt.subplots(len(idx), len(tags), figsize=(2.3 * len(tags), 2.4 * len(idx)), squeeze=False)
    for c, (tag, lab) in enumerate(zip(tags, labels)):
        for r, i in enumerate(idx):
            ax = axes[r, c]
            ax.imshow(IMAGES[tag][i]); ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(lab, fontsize=9)
            if c == 0:
                ax.set_ylabel(f"[{i}] " + PROMPTS[i][:28] + "…", fontsize=7)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout(); plt.savefig(PLOT_DIR / fname, dpi=110); plt.show()


def table(experiment):
    df = pd.DataFrame([r for r in RESULTS if r["experiment"] == experiment])
    cols = ["tag", "sampler", "steps", "cfg", "guidance_rescale", "clip_t", "clip_t_se", "clip_i", "sec_per_image"]
    return df[cols].round(4)

### Integration check 2 — images, not just embeddings

The same two prompts generated with `prompt=` (pipeline encodes) and with our `prompt_embeds` (we encode), from the
same latents. They should be visually identical; small pixel differences can come from fp16 non-determinism on
GPU.

In [ ]:
pipe.scheduler = make_scheduler(BASELINE["sampler"])
common = dict(num_inference_steps=BASELINE["steps"], guidance_scale=BASELINE["cfg"],
              latents=LATENTS[:2].to(DEVICE, SD_DTYPE), output_type="np")
via_prompt = pipe(prompt=PROMPTS[:2], **common).images
via_embeds = pipe(prompt_embeds=PROMPT_EMBEDS[:2], negative_prompt_embeds=EMPTY_EMBED.expand(2, -1, -1), **common).images
pix_diff = np.abs(via_prompt - via_embeds)
print(f"pixel difference (0–1 scale): mean {pix_diff.mean():.5f}  max {pix_diff.max():.5f}")

fig, axes = plt.subplots(2, 2, figsize=(6.5, 6.5))
for r in range(2):
    axes[r, 0].imshow(via_prompt[r]); axes[r, 1].imshow(via_embeds[r])
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_title("prompt= (pipeline encodes)", fontsize=9); axes[0, 1].set_title("prompt_embeds= (we encode)", fontsize=9)
plt.tight_layout(); plt.savefig(PLOT_DIR / "integration_check.png", dpi=110); plt.show()

---
## 5. Warm-up

The first GPU call compiles kernels and allocates memory, which would inflate the first run's timing. One
throw-away call fixes that.

In [ ]:
pipe.scheduler = make_scheduler("DDIM")
_ = pipe(prompt_embeds=PROMPT_EMBEDS[:1], negative_prompt_embeds=EMPTY_EMBED, num_inference_steps=2,
         latents=LATENTS[:1].to(DEVICE, SD_DTYPE), output_type="latent")
print("warm-up done")

---
## 6. Baseline conditional generation

SD 1.5's out-of-the-box settings: PNDM sampler, 50 steps, guidance 7.5. Every tuned configuration is compared to
this row.

In [ ]:
base_row = run("baseline", "baseline", **BASELINE)

fig, axes = plt.subplots(2, N_PROMPTS, figsize=(2.1 * N_PROMPTS, 4.6), squeeze=False)
for i in range(N_PROMPTS):
    axes[0, i].imshow(REAL_IMAGES[i]); axes[1, i].imshow(IMAGES["baseline"][i])
    axes[0, i].set_title(f"[{i}]", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].text(-0.1, 0.5, "real", transform=axes[0, 0].transAxes, rotation=90, va="center", ha="right")
axes[1, 0].text(-0.1, 0.5, "baseline", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
fig.suptitle("Baseline generation (PNDM · 50 steps · CFG 7.5) vs the real COCO photos", fontsize=12)
plt.tight_layout(); plt.savefig(PLOT_DIR / "baseline_vs_real.png", dpi=110); plt.show()

---
## 7. Classifier-free guidance

Guidance mixes the conditional and unconditional predictions: `ε = ε_uncond + s · (ε_cond − ε_uncond)`. At `s = 1`
the caption has its plain conditional effect; larger `s` pushes harder toward the caption, at the cost of
saturated colours and lost detail at high values. `guidance_rescale = 0.7` (Lin et al., 2023) rescales the guided
prediction back toward the conditional one's standard deviation, which is meant to undo that over-saturation — it is
tested at the two highest CFG values.

Sampler fixed to DPM++ 2M at 25 steps so the sweep is fast; §8 checks the sampler separately.

In [ ]:
cfg_tags = []
for s in CFG_GRID:
    tag = f"cfg_{s:g}"
    run(tag, "cfg", TUNE_SAMPLER, TUNE_STEPS, s)
    cfg_tags.append(tag)
rescale_tags = []
for s in RESCALE_CFGS:
    tag = f"cfg_{s:g}_rescale"
    run(tag, "cfg_rescale", TUNE_SAMPLER, TUNE_STEPS, s, rescale=GUIDANCE_RESCALE)
    rescale_tags.append(tag)

In [ ]:
cfg_df = pd.DataFrame([r for r in RESULTS if r["experiment"] == "cfg"])
res_df = pd.DataFrame([r for r in RESULTS if r["experiment"] == "cfg_rescale"])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, m, name in zip(axes, ["clip_t", "clip_i"], ["CLIP-T (image ↔ caption)", "CLIP-I (image ↔ real photo)"]):
    ax.errorbar(cfg_df["cfg"], cfg_df[m], yerr=cfg_df[f"{m}_se"], marker="o", capsize=3, label="CFG")
    ax.errorbar(res_df["cfg"], res_df[m], yerr=res_df[f"{m}_se"], marker="s", ls="none", capsize=3,
                label=f"CFG + rescale {GUIDANCE_RESCALE}")
    if m == "clip_t":
        ax.axhline(REAL_CLIP_T.mean(), color="grey", ls="--", lw=1, label="real photos")
    ax.set_xlabel("guidance scale"); ax.set_title(name); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(PLOT_DIR / "cfg_sweep.png", dpi=110); plt.show()
table("cfg")

In [ ]:
show_grid(cfg_tags + rescale_tags, f"Guidance sweep — {TUNE_SAMPLER}, {TUNE_STEPS} steps",
          "cfg_grid.png", labels=[f"CFG {s:g}" for s in CFG_GRID] + [f"CFG {s:g} + rescale" for s in RESCALE_CFGS])

---
## 8. Samplers

The sampler (scheduler) decides how the noise level drops from step to step and how each denoising update is
taken. All seven share SD 1.5's trained β-schedule and start from the same latents, at CFG 7.5. Step counts follow
each sampler's usual operating point: first-order samplers (Euler, DDIM) need more steps than the multistep
solvers (DPM++ 2M, UniPC). **Euler-a** is *ancestral*: it adds fresh noise every step, so it never converges to a
single image as steps grow.

In [ ]:
sampler_tags = []
for name, steps in SAMPLER_STEPS.items():
    tag = f"sampler_{name}"
    run(tag, "sampler", name, steps, 7.5)
    sampler_tags.append(tag)

In [ ]:
smp_df = pd.DataFrame([r for r in RESULTS if r["experiment"] == "sampler"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(smp_df["sampler"], smp_df["clip_t"], yerr=smp_df["clip_t_se"], capsize=3, color="#4C72B0")
axes[0].axhline(REAL_CLIP_T.mean(), color="grey", ls="--", lw=1, label="real photos")
axes[0].set_ylim(smp_df["clip_t"].min() - 0.03, max(smp_df["clip_t"].max(), REAL_CLIP_T.mean()) + 0.02)
axes[0].set_title("CLIP-T by sampler (CFG 7.5)"); axes[0].legend(fontsize=8)
axes[1].bar(smp_df["sampler"], smp_df["sec_per_image"], color="#DD8452")
for x, (s, t) in enumerate(zip(smp_df["steps"], smp_df["sec_per_image"])):
    axes[1].text(x, t, f"{s} steps", ha="center", va="bottom", fontsize=8)
axes[1].set_title("seconds per image")
for ax in axes:
    ax.tick_params(axis="x", rotation=30); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(PLOT_DIR / "samplers.png", dpi=110); plt.show()
table("sampler")

In [ ]:
show_grid(sampler_tags, "Samplers at CFG 7.5 — same prompts, same starting noise", "sampler_grid.png",
          labels=[f"{n} ({s})" for n, s in SAMPLER_STEPS.items()])

---
## 9. Number of inference steps

More steps means a finer path from noise to image and proportionally more time. The question is where quality
stops improving. Tested for the multistep DPM++ 2M and for DDIM (first-order, deterministic at `eta = 0`).

In [ ]:
step_tags = {s: [] for s in STEP_SAMPLERS}
for name in STEP_SAMPLERS:
    for n in STEP_GRID:
        tag = f"steps_{name}_{n}"
        run(tag, "steps", name, n, 7.5)
        step_tags[name].append(tag)

In [ ]:
st_df = pd.DataFrame([r for r in RESULTS if r["experiment"] == "steps"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, g in st_df.groupby("sampler"):
    axes[0].errorbar(g["steps"], g["clip_t"], yerr=g["clip_t_se"], marker="o", capsize=3, label=name)
    axes[1].plot(g["steps"], g["sec_per_image"], marker="o", label=name)
axes[0].set_title("CLIP-T vs steps"); axes[1].set_title("seconds per image vs steps")
for ax in axes:
    ax.set_xlabel("inference steps"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(PLOT_DIR / "steps.png", dpi=110); plt.show()

show_grid(step_tags[STEP_SAMPLERS[0]], f"{STEP_SAMPLERS[0]} — steps", "steps_grid.png",
          labels=[f"{n} steps" for n in STEP_GRID])
table("steps")

---
## 10. Noise schedule

These are scheduler-config settings, not call arguments. All on Euler at 20 steps, CFG 7.5:

| Setting | What it changes |
|---|---|
| `timestep_spacing` = `leading` (SD 1.5 default) / `trailing` / `linspace` | which of the 1,000 training timesteps the 20 steps visit; `leading` never starts at pure noise |
| `use_karras_sigmas` | spaces the noise levels as in Karras et al. (2022) — denser near low noise, aimed at few-step sampling |
| `rescale_betas_zero_snr` + `trailing` + `guidance_rescale 0.7` | makes the last training step pure noise (Lin et al., 2023); the fix for "cannot make a truly dark/bright image". SD 1.5 was **not** trained this way, so this tests what the fix does to a model that never saw it |
| `beta_schedule` = `linear` / `squaredcos_cap_v2` | a different noise ramp than SD 1.5 was trained on — expected to *hurt*, showing that the trained schedule matters |

In [ ]:
SCHEDULE_VARIANTS = {
    "default (leading)": ({}, 0.0),
    "trailing": ({"timestep_spacing": "trailing"}, 0.0),
    "linspace": ({"timestep_spacing": "linspace"}, 0.0),
    "Karras sigmas": ({"use_karras_sigmas": True}, 0.0),
    "zero-SNR": ({"rescale_betas_zero_snr": True, "timestep_spacing": "trailing"}, GUIDANCE_RESCALE),
    "β linear": ({"beta_schedule": "linear"}, 0.0),
    "β cosine": ({"beta_schedule": "squaredcos_cap_v2"}, 0.0),
}
schedule_tags = []
for label, (kw, rescale) in SCHEDULE_VARIANTS.items():
    tag = "sched_" + label.split(" (")[0].replace(" ", "_").replace("β", "beta")
    run(tag, "schedule", SCHEDULE_SAMPLER, SCHEDULE_STEPS, 7.5, rescale=rescale, sched_kw=kw)
    schedule_tags.append(tag)

In [ ]:
sc_df = pd.DataFrame([r for r in RESULTS if r["experiment"] == "schedule"])
sc_df["variant"] = list(SCHEDULE_VARIANTS)
fig, ax = plt.subplots(figsize=(10, 3.6))
x = np.arange(len(sc_df))
ax.bar(x - 0.2, sc_df["clip_t"], 0.4, yerr=sc_df["clip_t_se"], capsize=3, label="CLIP-T")
ax.bar(x + 0.2, sc_df["clip_i"], 0.4, yerr=sc_df["clip_i_se"], capsize=3, label="CLIP-I")
ax.set_xticks(x); ax.set_xticklabels(sc_df["variant"], rotation=20)
ax.set_title(f"Noise-schedule variants — {SCHEDULE_SAMPLER}, {SCHEDULE_STEPS} steps, CFG 7.5")
ax.grid(axis="y", alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(PLOT_DIR / "schedule.png", dpi=110); plt.show()

show_grid(schedule_tags, "Noise-schedule variants", "schedule_grid.png", labels=list(SCHEDULE_VARIANTS))
sc_df[["variant", "clip_t", "clip_t_se", "clip_i", "sec_per_image"]].round(4)

---
## 11. Negative prompts

With guidance, the negative prompt replaces the empty string in the unconditional half, so the sampler is pushed
*away* from it. Two kinds are tried: a **quality** negative (blur, artefacts, watermarks) and a **style** negative
that steers away from photorealism — a simple form of style control with no training.

In [ ]:
neg_tags = [f"cfg_{7.5:g}"]          # the no-negative run is already in the CFG sweep (same sampler, steps, CFG)
for name, text in NEGATIVE_PROMPTS.items():
    tag = f"neg_{name}"
    run(tag, "negative", TUNE_SAMPLER, TUNE_STEPS, 7.5, negative=text)
    neg_tags.append(tag)

neg_df = pd.DataFrame([r for r in RESULTS if r["tag"] in neg_tags]).set_index("tag").loc[neg_tags]
neg_df.insert(0, "negative prompt", ["(none)"] + list(NEGATIVE_PROMPTS))
display(neg_df[["negative prompt", "clip_t", "clip_t_se", "clip_i", "clip_i_se"]].round(4))
show_grid(neg_tags, f"Negative prompts — {TUNE_SAMPLER}, {TUNE_STEPS} steps, CFG 7.5", "negative_grid.png",
          labels=["no negative", "quality negative", "style negative"])

### DDPM: 1,000 steps vs 20

DDPM is the original sampler the model was trained with: one step per training timestep, so 1,000 UNet calls per
image. It is left out of the sweep for cost and run once on 2 prompts to show what modern solvers buy: roughly the
same picture from the same noise in 2 % of the steps.

In [ ]:
ddpm_idx = list(range(min(DDPM_PROMPTS, N_PROMPTS)))
run("ddpm_1000", "ddpm", "DDPM", DDPM_STEPS, 7.5, idx=ddpm_idx)
run("ddpm_vs_dpm20", "ddpm", "DPM++ 2M", 20 if not SMOKE_TEST else 4, 7.5, idx=ddpm_idx)
show_grid(["ddpm_1000", "ddpm_vs_dpm20"], "Same noise, same prompt — DDPM vs DPM++ 2M", "ddpm_grid.png",
          labels=[f"DDPM, {DDPM_STEPS} steps", "DPM++ 2M, 20 steps"], idx=ddpm_idx)
table("ddpm")

---
## 12. Quality vs. speed

Every run from §6–§11 (full 10-prompt runs only) on one chart. The upper-left frontier — the best CLIP-T reachable
for a given time per image — is where a practical setting should come from.

In [ ]:
all_df = pd.DataFrame(RESULTS)
full = all_df[all_df["n"] == N_PROMPTS].copy()

fig, ax = plt.subplots(figsize=(10, 5))
for exp, g in full.groupby("experiment"):
    ax.scatter(g["sec_per_image"], g["clip_t"], label=exp, s=40, alpha=.8)
frontier = full.sort_values("sec_per_image")
frontier = frontier[frontier["clip_t"] >= frontier["clip_t"].cummax()]
ax.plot(frontier["sec_per_image"], frontier["clip_t"], "k--", lw=1, label="frontier")
for _, r in frontier.iterrows():
    ax.annotate(r["tag"], (r["sec_per_image"], r["clip_t"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.axhline(REAL_CLIP_T.mean(), color="grey", ls=":", lw=1, label="real photos")
ax.set_xlabel("seconds per image"); ax.set_ylabel("CLIP-T"); ax.grid(alpha=.3); ax.legend(fontsize=8)
ax.set_title("Prompt adherence vs. generation time — every configuration")
plt.tight_layout(); plt.savefig(PLOT_DIR / "quality_vs_speed.png", dpi=110); plt.show()
frontier[["tag", "sampler", "steps", "cfg", "clip_t", "clip_i", "sec_per_image"]].round(4)

---
## 13. Baseline vs. best

"Best" is the highest CLIP-T among the full 10-prompt runs; "fastest good" is the quickest run whose CLIP-T is
within one standard error of the best. With 10 prompts the standard errors are wide, so differences smaller than
about one s.e. should be read as ties, not wins.

In [ ]:
best = full.loc[full["clip_t"].idxmax()]
fast_good = full[full["clip_t"] >= best["clip_t"] - best["clip_t_se"]].sort_values("sec_per_image").iloc[0]
picks = {"baseline": base_row, "best CLIP-T": best, "fastest within 1 s.e.": fast_good}

summary = pd.DataFrame({k: {c: v[c] for c in ["tag", "sampler", "steps", "cfg", "guidance_rescale", "negative",
                                              "schedule", "clip_t", "clip_t_se", "clip_i", "sec_per_image"]}
                        for k, v in picks.items()}).T
summary.loc["real photos", ["clip_t", "clip_t_se"]] = [REAL_CLIP_T.mean(), REAL_CLIP_T.std(ddof=1) / np.sqrt(N_PROMPTS)]
display(summary)

grid_labels = {}                     # one column per distinct run; a run picked twice gets both names
for k, v in picks.items():
    grid_labels.setdefault(v["tag"], []).append(k)
show_grid(list(grid_labels), "Baseline vs. tuned", "baseline_vs_best.png",
          labels=[" / ".join(v) + f"\n{t}" for t, v in grid_labels.items()], n_rows=N_PROMPTS)

---
## 14. Save everything

`results.csv` (one row per run), `summary.csv`, `metadata.json`, every PNG under `generations/<run>/` and every
figure under `plots/`, zipped for download. On Colab the zip is offered as a download; it can also be copied to
Google Drive.

In [ ]:
all_df.round(5).to_csv(OUT_DIR / "results.csv", index=False)
summary.to_csv(OUT_DIR / "summary.csv")
prompts_df[["image_id", "ann_id", "file_name", "caption"]].to_csv(OUT_DIR / "prompts.csv", index_label="prompt_idx")

metadata = {
    "versions": VERSIONS, "sd_repo": SD_REPO, "eval_model": EVAL_MODEL, "dataset": KAGGLE_DATASET,
    "seed": SEED, "n_prompts": N_PROMPTS, "latent_shape": list(LATENT_SHAPE), "sd_dtype": str(SD_DTYPE),
    "baseline": BASELINE, "cfg_grid": CFG_GRID, "sampler_steps": SAMPLER_STEPS, "step_grid": STEP_GRID,
    "schedule_variants": {k: v[0] for k, v in SCHEDULE_VARIANTS.items()}, "negative_prompts": NEGATIVE_PROMPTS,
    "real_clip_t": float(REAL_CLIP_T.mean()), "n_runs": len(RESULTS),
    "total_generation_seconds": float((all_df["sec_per_image"] * all_df["n"]).sum()),
    "smoke_test": SMOKE_TEST,
}
(OUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str))

archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print(f"{len(RESULTS)} runs · {metadata['total_generation_seconds'] / 60:.1f} min of generation")
print("saved:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)

---
## 15. Summary

The cell below prints the headline numbers from this run. The observations to look for in the figures above:

- **CFG (§7):** where CLIP-T peaks, whether it falls off at high CFG, and whether `guidance_rescale` recovers it.
- **Samplers (§8):** whether the 20-step multistep solvers match the 50-step PNDM baseline at less than half the
  time.
- **Steps (§9):** the step count past which CLIP-T stops improving.
- **Schedule (§10):** whether the alternative β-schedules, which SD 1.5 was not trained on, degrade the images.
- **Negative prompts (§11):** the quality negative should change little in CLIP-T; the style negative should
  visibly move away from photographs, and CLIP-I (closeness to the real photo) should drop.

### Next steps (Milestone 3)

- FID and Inception Score for the baseline and the two or three best configurations, on a few thousand prompts
  rather than 10 (CLIP-T on 10 prompts ranks settings; it cannot separate close ones).
- CLIP-T and CLIP-I for the whole curated set under the chosen setting, with failure-case analysis on the
  lowest-scoring prompts.

In [ ]:
def line(label, r):
    return (f"{label:<22} {r['tag']:<26} CLIP-T {r['clip_t']:.4f} ± {r['clip_t_se']:.4f}   "
            f"CLIP-I {r['clip_i']:.4f}   {r['sec_per_image']:.2f} s/img")

print(f"{'real photos':<22} {'':<26} CLIP-T {REAL_CLIP_T.mean():.4f}")
for k, v in picks.items():
    print(line(k, v))
cfg_best = cfg_df.loc[cfg_df["clip_t"].idxmax()]
print(f"\nbest guidance scale    : {cfg_best['cfg']:g} (CLIP-T {cfg_best['clip_t']:.4f})")
print(f"best sampler           : {smp_df.loc[smp_df['clip_t'].idxmax(), 'sampler']}")
print(f"fastest sampler        : {smp_df.loc[smp_df['sec_per_image'].idxmin(), 'sampler']}")
speedup = base_row["sec_per_image"] / fast_good["sec_per_image"]
print(f"speed-up vs baseline   : {speedup:.1f}× ({fast_good['tag']})")